# PomBase — *Schizosaccharomyces pombe* Genome Database

**PomBase** is the model organism database (MOD) for the fission yeast *Schizosaccharomyces pombe*. It provides comprehensive genomic annotation, experimental data, and community curation for one of the most important eukaryotic model organisms in cell biology research.

| Property | Value |
|---|---|
| URL | https://www.pombase.org |
| Genome size | ~12.6 Mb |
| Protein-coding genes | ~7,000 |
| Primary use cases | Cell cycle, chromosome biology, gene regulation |

In [ ]:
import requests
import time
import gzip
import io
from pathlib import Path

import polars as pl
import pandas as pd

# TODO

* [x] **Ingest data**
    * [x] Connect to PomBase REST API and confirm access
    * [x] Download gene IDs/names/products TSV from PomBase bulk data with caching
    * [x] Download GO annotation GAF file from PomBase with caching
    * [x] Parse both files into Polars DataFrames with correct dtypes
    * [x] Save to `data/` with caching
* [ ] **Explore and clean**
    * [ ] Summarise gene annotation completeness (% with standard names, GO terms)
    * [ ] Check for duplicate systematic IDs
    * [ ] Parse GO term categories (molecular function, biological process, cellular component)
    * [ ] Filter out pseudogenes and non-coding RNA loci
* [ ] **Analysis**
    * [ ] GO term frequency distribution
    * [ ] Identify most/least annotated genes
    * [ ] Compare annotation depth across chromosomes
* [ ] **Visualization**
    * [ ] Bar chart of GO term categories
    * [ ] Histogram of GO terms per gene
* [ ] **Statistical analysis**
    * [ ] Test for non-uniform GO category enrichment
    * [ ] Compare annotation completeness with other model organisms

## 1. Ingest Data

### 1.1 Connect to PomBase API

In [ ]:
POMBASE_API = "https://www.pombase.org/api/v1"

def pombase_get(endpoint: str, params: dict | None = None) -> dict:
    """
    Send a GET request to the PomBase REST API.

    Parameters
    ----------
    endpoint : str
        API path relative to the v1 base, e.g. "dataset/gene/SPBC11C11.03".
    params : dict, optional
        Additional query parameters.

    Returns
    -------
    dict
        Parsed JSON response body.
    """
    url = f"{POMBASE_API}/{endpoint}"
    resp = requests.get(url, params=params or {}, timeout=30)
    resp.raise_for_status()
    return resp.json()

# Confirm access: look up cdc2 (cyclin-dependent kinase 1), the master cell-cycle regulator
# Systematic ID: SPBC11C11.03
CDC2_ID = "SPBC11C11.03"
gene_data = pombase_get(f"dataset/gene/{CDC2_ID}")

print(f"Gene:        {gene_data.get('uniquename', 'N/A')}")
print(f"Name:        {gene_data.get('name', 'N/A')}")
print(f"Product:     {gene_data.get('product', 'N/A')}")
print(f"Chromosome:  {gene_data.get('location', {}).get('chromosome_name', 'N/A')}")
print(f"Strand:      {gene_data.get('location', {}).get('strand', 'N/A')}")
print(f"Start:       {gene_data.get('location', {}).get('start_pos', 'N/A')}")
print(f"End:         {gene_data.get('location', {}).get('end_pos', 'N/A')}")

### 1.2 Download Gene IDs and Names

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Bulk TSV: all gene IDs, systematic names, standard names, and product descriptions
GENE_IDS_URL = "https://www.pombase.org/data/names_and_identifiers/gene_IDs_names_products.tsv"
GENE_IDS_PATH = DATA_DIR / "gene_IDs_names_products.tsv"

if not GENE_IDS_PATH.exists():
    print(f"Downloading {GENE_IDS_PATH.name} ...")
    with requests.get(GENE_IDS_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(GENE_IDS_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"  {downloaded / 1e3:.1f} / {total / 1e3:.1f} KB", end="\r")
    print(f"\nSaved to {GENE_IDS_PATH}")
else:
    print(f"Already downloaded: {GENE_IDS_PATH}")

### 1.3 Download GO Annotations

In [ ]:
# GO annotations in GAF 2.2 format (gzipped)
GAF_URL = "https://www.pombase.org/data/annotations/gene_ontology/go_style_gaf/pombase.gaf.gz"
GAF_PATH = DATA_DIR / "pombase.gaf.gz"

if not GAF_PATH.exists():
    print(f"Downloading {GAF_PATH.name} ...")
    with requests.get(GAF_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(GAF_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    print(f"  {downloaded / 1e6:.1f} / {total / 1e6:.1f} MB", end="\r")
    print(f"\nSaved to {GAF_PATH}")
else:
    print(f"Already downloaded: {GAF_PATH}")

### 1.4 Parse Gene Table into Polars DataFrame

In [ ]:
# The TSV has a header line beginning with '#' — use comment_prefix to skip it.
# Columns (as documented by PomBase):
#   systematic_id, primary_name, product_description, uniprot_id,
#   chromosome, start, end, strand
genes = pl.read_csv(
    GENE_IDS_PATH,
    separator="\t",
    comment_prefix="!",
    has_header=True,
    schema_overrides={
        "start_position": pl.Int32,
        "end_position": pl.Int32,
    },
    infer_schema_length=10_000,
)

# Normalise column names to snake_case
genes.columns = [c.lower().replace(" ", "_") for c in genes.columns]

print(f"Shape: {genes.shape}")
print(f"Columns: {genes.columns}")
genes.head(5)

### 1.5 Parse GAF into Polars DataFrame

In [ ]:
# GAF 2.2 standard column names (17 columns, tab-separated)
# Lines beginning with '!' are header/comment lines and must be skipped.
GAF_COLUMNS = [
    "db",                    # col 1  — database abbreviation (e.g. "PomBase")
    "db_object_id",          # col 2  — gene/protein ID in source database
    "db_object_symbol",      # col 3  — gene symbol (e.g. "cdc2")
    "qualifier",             # col 4  — optional modifier (e.g. "NOT")
    "go_id",                 # col 5  — GO term accession (e.g. "GO:0000287")
    "db_reference",          # col 6  — reference (PMID or GO_REF)
    "evidence_code",         # col 7  — evidence code (e.g. "IDA", "IMP", "IEA")
    "with_from",             # col 8  — with/from field for IEA/ISS etc.
    "aspect",                # col 9  — ontology namespace: F, P, or C
    "db_object_name",        # col 10 — full gene/protein name
    "db_object_synonym",     # col 11 — pipe-separated synonyms
    "db_object_type",        # col 12 — gene, protein, etc.
    "taxon",                 # col 13 — NCBI taxon ID (e.g. "taxon:4896")
    "date",                  # col 14 — annotation date (YYYYMMDD)
    "assigned_by",           # col 15 — database that made the annotation
    "annotation_extension",  # col 16 — optional relation:ID extensions
    "gene_product_form_id",  # col 17 — specific isoform/splice variant
]

# Read the gzip file in Python, strip comment lines, pass bytes to Polars
with gzip.open(GAF_PATH, "rb") as gz:
    raw_bytes = gz.read()

# Filter out comment/header lines (those starting with '!')
data_lines = [
    line for line in raw_bytes.split(b"\n")
    if line and not line.startswith(b"!")
]
clean_bytes = b"\n".join(data_lines)

gaf = pl.read_csv(
    io.BytesIO(clean_bytes),
    separator="\t",
    has_header=False,
    new_columns=GAF_COLUMNS,
    infer_schema_length=10_000,
)

print(f"Shape: {gaf.shape}")
print(f"Columns: {gaf.columns}")
gaf.head(5)